In [1]:
import pandas as pd
import numpy as np
import anthropic
import os
from dotenv import load_dotenv
import time
import json

# Load API key from .env file
load_dotenv("../.env")
api_key = os.getenv("ANTHROPIC_API_KEY")

if api_key:
    print("API key loaded ✓")
else:
    print("ERROR — API key not found. Check your .env file")

API key loaded ✓


In [2]:
# import sys
#!{sys.executable} -m pip install anthropic python-dotenv

In [3]:
client = anthropic.Anthropic(api_key=api_key)

def rate_comment_with_claude(comment_text):
    
    prompt = f"""You are a Trust & Safety analyst reviewing user-generated content for policy violations.

Analyze the following comment and return a JSON response with exactly these fields:

{{
    "policy_category": one of ["hate_speech", "harassment", "sexual_content", "self_harm", "spam", "fraud", "safe"],
    "severity": one of ["none", "low", "medium", "high", "critical"],
    "action": one of ["allow", "downrank", "human_review", "auto_block"],
    "reasoning": "one sentence explanation"
}}

Comment to analyze:
\"\"\"{comment_text}\"\"\"

Return only the JSON object, no other text."""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    
    # Print raw response so we can see what Claude returned
    raw = response.content[0].text
    print("Raw response:", raw)
    
    # Clean and parse
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()
    
    result = json.loads(raw)
    return result

# Test with one comment
test_comment = "You are the stupidest person I have ever seen. I hope you die."

result = rate_comment_with_claude(test_comment)
print("Claude's rating:")
print(json.dumps(result, indent=2))

Raw response: ```json
{
    "policy_category": "harassment",
    "severity": "high",
    "action": "auto_block",
    "reasoning": "This comment contains a direct personal attack with a death wish, which constitutes severe harassment and violates safety policies."
}
```
Claude's rating:
{
  "policy_category": "harassment",
  "severity": "high",
  "action": "auto_block",
  "reasoning": "This comment contains a direct personal attack with a death wish, which constitutes severe harassment and violates safety policies."
}


In [4]:
# Load eval dataframe we saved earlier
df_eval = pd.read_csv("../models/df_eval.csv")

# Strategically sample 200 comments
# 1. High confidence toxic (ML says HIGH, true label = 1)
high_toxic = df_eval[
    (df_eval["ml_bucket"] == "HIGH") & 
    (df_eval["true_label"] == 1)
].sample(60, random_state=42)

# 2. Edge cases — ML says MED but true label = 1
edge_cases = df_eval[
    (df_eval["ml_bucket"] == "MED") & 
    (df_eval["true_label"] == 1)
].sample(40, random_state=42)

# 3. False negatives — ML says SAFE but true label = 1 (missed toxic)
false_negatives = df_eval[
    (df_eval["ml_bucket"] == "SAFE") & 
    (df_eval["true_label"] == 1)
].sample(50, random_state=42)

# 4. Clean comments as control
clean = df_eval[
    (df_eval["ml_bucket"] == "SAFE") & 
    (df_eval["true_label"] == 0)
].sample(50, random_state=42)

# Combine
df_sample = pd.concat([high_toxic, edge_cases, false_negatives, clean])
df_sample = df_sample.reset_index(drop=True)

print("Sample breakdown:")
print(f"  High toxic:      {len(high_toxic)}")
print(f"  Edge cases:      {len(edge_cases)}")
print(f"  False negatives: {len(false_negatives)}")
print(f"  Clean control:   {len(clean)}")
print(f"  Total:           {len(df_sample)}")

Sample breakdown:
  High toxic:      60
  Edge cases:      40
  False negatives: 50
  Clean control:   50
  Total:           200


In [5]:
def rate_comment_safe(comment_text, idx):
    """Rate a comment with Claude, with error handling"""
    try:
        result = rate_comment_with_claude(comment_text)
        result["idx"] = idx
        return result
    except Exception as e:
        print(f"Error on row {idx}: {e}")
        return {
            "idx": idx,
            "policy_category": "error",
            "severity": "none",
            "action": "allow",
            "reasoning": f"API error: {str(e)}"
        }

# Run Claude on all 200 comments
results = []
total = len(df_sample)

for i, row in df_sample.iterrows():
    result = rate_comment_safe(row["comment_text"], i)
    results.append(result)
    
    # Progress update every 20 comments
    if (len(results)) % 20 == 0:
        print(f"Progress: {len(results)}/{total} done...")
    
    # Small delay to avoid rate limits
    time.sleep(0.5)

print(f"\nDone! Rated {len(results)} comments")

Raw response: ```json
{
    "policy_category": "harassment",
    "severity": "low",
    "action": "allow",
    "reasoning": "User is defending themselves against accusations with frustrated language and profanity but without targeted personal attacks or threats."
}
```
Raw response: ```json
{
    "policy_category": "harassment",
    "severity": "critical",
    "action": "auto_block",
    "reasoning": "This comment contains direct harassment including a suicide directive, personal attacks, and vulgar insults targeting the recipient and their family member."
}
```
Raw response: ```json
{
    "policy_category": "harassment",
    "severity": "high",
    "action": "auto_block",
    "reasoning": "Content contains multiple homophobic slurs, threatening language about IP tracing, and severe personal attacks that constitute clear harassment."
}
```
Raw response: ```json
{
    "policy_category": "safe",
    "severity": "none",
    "action": "allow",
    "reasoning": "This is a harmless self-depr

In [6]:
# Convert results list to dataframe
df_claude = pd.DataFrame(results)

# Merge with original sample
df_sample = df_sample.reset_index(drop=True)
df_claude = df_claude.rename(columns={
    "policy_category": "claude_category",
    "severity": "claude_severity",
    "action": "claude_action",
    "reasoning": "claude_reasoning"
})

# Combine
df_combined = pd.concat([df_sample.reset_index(drop=True), 
                          df_claude.drop(columns=["idx"])], axis=1)

# Map claude action to bucket for comparison
def action_to_bucket(action):
    if action == "auto_block": return "HIGH"
    if action == "human_review": return "MED"
    if action == "downrank": return "LOW"
    return "SAFE"

df_combined["claude_bucket"] = df_combined["claude_action"].apply(action_to_bucket)

# Disagreement flag
df_combined["disagree"] = df_combined["ml_bucket"] != df_combined["claude_bucket"]

print("Combined dataframe shape:", df_combined.shape)
print("\nClaude policy category distribution:")
print(df_combined["claude_category"].value_counts())
print("\nDisagreement rate:", round(df_combined["disagree"].mean(), 4))

Combined dataframe shape: (200, 11)

Claude policy category distribution:
claude_category
harassment        98
safe              77
hate_speech       16
spam               4
sexual_content     3
self_harm          2
Name: count, dtype: int64

Disagreement rate: 0.485


In [7]:
# Disagreement breakdown
print("=== DISAGREEMENT ANALYSIS ===\n")

disagree_df = df_combined[df_combined["disagree"] == True].copy()
agree_df = df_combined[df_combined["disagree"] == False].copy()

print(f"Total comments: {len(df_combined)}")
print(f"Agreements: {len(agree_df)} ({round(len(agree_df)/len(df_combined)*100,1)}%)")
print(f"Disagreements: {len(disagree_df)} ({round(len(disagree_df)/len(df_combined)*100,1)}%)")

print("\n--- Where they disagree ---")
print(pd.crosstab(df_combined["ml_bucket"], df_combined["claude_bucket"], 
                   rownames=["ML"], colnames=["Claude"]))

print("\n--- ML says HIGH but Claude says SAFE (ML over-flagging) ---")
ml_high_claude_safe = df_combined[
    (df_combined["ml_bucket"] == "HIGH") & 
    (df_combined["claude_bucket"] == "SAFE")
]
print(f"Count: {len(ml_high_claude_safe)}")
for _, row in ml_high_claude_safe.head(3).iterrows():
    print(f"\nComment: {row['comment_text'][:100]}")
    print(f"Claude reasoning: {row['claude_reasoning']}")

print("\n--- ML says SAFE but Claude flags (ML missing toxic) ---")
ml_safe_claude_flags = df_combined[
    (df_combined["ml_bucket"] == "SAFE") & 
    (df_combined["claude_bucket"].isin(["MED", "HIGH"]))
]
print(f"Count: {len(ml_safe_claude_flags)}")
for _, row in ml_safe_claude_flags.head(3).iterrows():
    print(f"\nComment: {row['comment_text'][:100]}")
    print(f"Claude category: {row['claude_category']} | Severity: {row['claude_severity']}")
    print(f"Claude reasoning: {row['claude_reasoning']}")

=== DISAGREEMENT ANALYSIS ===

Total comments: 200
Agreements: 103 (51.5%)
Disagreements: 97 (48.5%)

--- Where they disagree ---
Claude  HIGH  LOW  MED  SAFE
ML                          
HIGH      29    6   17     8
MED       10    6   13    11
SAFE       8   10   21    61

--- ML says HIGH but Claude says SAFE (ML over-flagging) ---
Count: 8

Comment: "|""ugly comments"" are a crime? Attacking religious beliefs? Bullshit. And ""page blankings""?? I n
Claude reasoning: User is defending themselves against accusations with frustrated language and profanity but without targeted personal attacks or threats.

Comment: and yeah i suck at wiki formatting.......
Claude reasoning: This is a harmless self-deprecating comment about the user's own wiki formatting skills with no policy violations.

Comment: "They aren't ""questionable"".  READ the article, it says ""Jacobson cursed the University of Notre 
Claude reasoning: The comment discusses a news story about profanity used by a public figur

In [8]:
print("=== FALSE NEGATIVE ANALYSIS ===\n")
print("(Toxic comments that slipped through BOTH ML and Claude)\n")

# Comments that are truly toxic but both systems missed
both_missed = df_combined[
    (df_combined["true_label"] == 1) &
    (df_combined["ml_bucket"] == "SAFE") &
    (df_combined["claude_bucket"] == "SAFE")
]

# Claude catches what ML missed
claude_saves = df_combined[
    (df_combined["true_label"] == 1) &
    (df_combined["ml_bucket"] == "SAFE") &
    (df_combined["claude_bucket"].isin(["MED", "HIGH"]))
]

# ML catches what Claude missed
ml_saves = df_combined[
    (df_combined["true_label"] == 1) &
    (df_combined["claude_bucket"] == "SAFE") &
    (df_combined["ml_bucket"].isin(["MED", "HIGH"]))
]

total_toxic = df_combined["true_label"].sum()

print(f"Total toxic in sample: {total_toxic}")
print(f"Both missed (critical escapes): {len(both_missed)} ({round(len(both_missed)/total_toxic*100,1)}%)")
print(f"Claude saved what ML missed: {len(claude_saves)}")
print(f"ML saved what Claude missed: {len(ml_saves)}")

print("\n--- Examples Claude caught that ML missed ---")
for _, row in claude_saves.head(3).iterrows():
    print(f"\nComment: {row['comment_text'][:120]}")
    print(f"Claude: {row['claude_category']} | {row['claude_severity']} | {row['claude_reasoning']}")

=== FALSE NEGATIVE ANALYSIS ===

(Toxic comments that slipped through BOTH ML and Claude)

Total toxic in sample: 150
Both missed (critical escapes): 15 (10.0%)
Claude saved what ML missed: 28
ML saved what Claude missed: 19

--- Examples Claude caught that ML missed ---

Comment: "

WRONG retard, The good ending is not the canon ending. Wheeler only said that no one knows what happened to Cybil. Sh
Claude: harassment | medium | Comment contains an ableist slur ('retard') directed at another user in a disagreement about video game lore, which constitutes harassing language despite otherwise substantive content.

Comment: FUK-U DANIEL CASE AND KATIE DUM BICH
Claude: harassment | medium | Direct personal attack targeting named individuals with profanity and derogatory language constitutes harassment.

Comment: FURTHERMORE.... I HAVE JUST VISITED RAGIB'S PAGE AND STUDIED THE DISCUSSION AREA. RAGIB IS OBVIOUSLY FROM BANGLADESH AND
Claude: harassment | medium | Comment involves targeting a 

In [9]:
# Save combined results
df_combined.to_csv("../models/df_combined.csv", index=False)

# Save disagreement cases separately — useful for Streamlit
disagree_df.to_csv("../models/df_disagree.csv", index=False)

# Save false negatives
both_missed.to_csv("../models/df_missed.csv", index=False)

print("Saved:")
print(f"  df_combined.csv     — {len(df_combined)} rows")
print(f"  df_disagree.csv     — {len(disagree_df)} rows")
print(f"  df_missed.csv       — {len(both_missed)} rows")

# Print final summary
print("\n=== FINAL SUMMARY ===")
print(f"Total comments analyzed: {len(df_combined)}")
print(f"Disagreement rate (ML vs Claude): {round(df_combined['disagree'].mean()*100,1)}%")
print(f"Critical escape rate: {round(len(both_missed)/total_toxic*100,1)}%")
print(f"Claude saved from ML misses: {len(claude_saves)}")
print(f"ML saved from Claude misses: {len(ml_saves)}")
print(f"\nPolicy categories Claude detected:")
print(df_combined['claude_category'].value_counts().to_string())

Saved:
  df_combined.csv     — 200 rows
  df_disagree.csv     — 97 rows
  df_missed.csv       — 15 rows

=== FINAL SUMMARY ===
Total comments analyzed: 200
Disagreement rate (ML vs Claude): 48.5%
Critical escape rate: 10.0%
Claude saved from ML misses: 28
ML saved from Claude misses: 19

Policy categories Claude detected:
claude_category
harassment        98
safe              77
hate_speech       16
spam               4
sexual_content     3
self_harm          2
